# SERENE AI — Phase 2: Facial Emotion Model Training & Comparison
## Comparing MobileNetV3 vs. Vision Transformer (ViT) on FER-2013

**Objective**:
- Benchmark **MobileNetV3-Large** (~5.4M params) against **Vision Transformer (ViT-Base-Patch16)** (~86M params) on FER-2013.
- Address extreme class imbalance (e.g. Happy 5,772 vs. Disgust 349) using `WeightedRandomSampler` and class-weighted CrossEntropyLoss.
- Evaluate overall Accuracy, Macro-F1, and generate a 7x7 normalized Confusion Matrix.
- Export the optimal model to `checkpoints/face_model/` for production inference.

In [ ]:
# 1. Environment Setup & Library Installation
!nvidia-smi
!pip install -q timm transformers torchvision scikit-learn matplotlib seaborn pandas tqdm

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 2. Load Manifest & Inspect Class Imbalance
# When running locally or on Colab, point to fer2013_manifest.csv
MANIFEST_PATH = "fer2013_manifest.csv"
df = pd.read_csv(MANIFEST_PATH)

EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
NUM_CLASSES = 7

print("Split distribution:")
print(df["split"].value_counts())

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

# Compute Class Weights for CrossEntropyLoss
class_counts = train_df["label"].value_counts().sort_index().values
total_train = len(train_df)
class_weights = total_train / (NUM_CLASSES * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("\nComputed Inverse Class Weights:")
for i, emo in enumerate(EMOTIONS):
    print(f"  {emo:8s} ({class_counts[i]:4d} samples): weight = {class_weights[i]:.3f}")

In [ ]:
# 3. PyTorch Dataset & Weighted Sampler
class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx]["absolute_path"]
        image = Image.open(path).convert("RGB")
        label = int(self.df.iloc[idx]["label"])
        if self.transform:
            image = self.transform(image)
        return image, label

# Create WeightedRandomSampler for training
sample_weights = [class_weights[int(l)] for l in train_df["label"]]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# Transformations for MobileNetV3 (48x48 input)
mobilenet_train_tx = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
mobilenet_eval_tx = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transformations for ViT (224x224 input)
vit_train_tx = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
vit_eval_tx = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
# 4. Generic Training & Evaluation Helper
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, model_name):
    scaler = torch.cuda.amp.GradScaler()
    best_f1 = 0.0
    best_weights = None
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for imgs, lbls in tqdm(train_loader, desc=f"{model_name} Epoch {epoch+1}/{num_epochs}"):
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                loss = criterion(logits, lbls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
        
        if scheduler:
            scheduler.step()
            
        # Validate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                preds = torch.argmax(logits, dim=-1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())
                
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        print(f"{model_name} Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = model.state_dict()
            
    model.load_state_dict(best_weights)
    return model, best_f1

In [ ]:
# 5. Train & Evaluate Model 1: MobileNetV3-Large
mb_train_ds = FERDataset(train_df, mobilenet_train_tx)
mb_val_ds   = FERDataset(val_df, mobilenet_eval_tx)
mb_test_ds  = FERDataset(test_df, mobilenet_eval_tx)

mb_train_loader = DataLoader(mb_train_ds, batch_size=64, sampler=sampler)
mb_val_loader   = DataLoader(mb_val_ds, batch_size=64, shuffle=False)
mb_test_loader  = DataLoader(mb_test_ds, batch_size=64, shuffle=False)

mobilenet = models.mobilenet_v3_large(weights='DEFAULT')
mobilenet.classifier[3] = nn.Linear(mobilenet.classifier[3].in_features, NUM_CLASSES)
mobilenet = mobilenet.to(device)

mb_criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
mb_optimizer = torch.optim.AdamW(mobilenet.parameters(), lr=1e-4, weight_decay=1e-4)
mb_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(mb_optimizer, T_max=15)

mobilenet, mb_best_f1 = train_model(
    mobilenet, mb_train_loader, mb_val_loader,
    mb_criterion, mb_optimizer, mb_scheduler, num_epochs=15, model_name="MobileNetV3"
)

In [ ]:
# 6. Train & Evaluate Model 2: Vision Transformer (ViT-Base-Patch16)
vit_train_ds = FERDataset(train_df, vit_train_tx)
vit_val_ds   = FERDataset(val_df, vit_eval_tx)
vit_test_ds  = FERDataset(test_df, vit_eval_tx)

vit_train_loader = DataLoader(vit_train_ds, batch_size=32, sampler=sampler)
vit_val_loader   = DataLoader(vit_val_ds, batch_size=32, shuffle=False)
vit_test_loader  = DataLoader(vit_test_ds, batch_size=32, shuffle=False)

id2label = {i: name for i, name in enumerate(EMOTIONS)}
label2id = {name: i for i, name in enumerate(EMOTIONS)}

vit_model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id
).to(device)

vit_criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
vit_optimizer = torch.optim.AdamW(vit_model.parameters(), lr=2e-5, weight_decay=0.01)

vit_model, vit_best_f1 = train_model(
    vit_model, vit_train_loader, vit_val_loader,
    vit_criterion, vit_optimizer, scheduler=None, num_epochs=10, model_name="VisionTransformer"
)

In [ ]:
# 7. Comparative Evaluation & Confusion Matrix on Test Set
def evaluate_and_plot(model, loader, model_name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs = model(imgs)
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds, normalize='true')
    
    print(f"\n=== {model_name} Test Results ===")
    print(f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
    print(classification_report(all_labels, all_preds, target_names=EMOTIONS))
    
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=EMOTIONS, yticklabels=EMOTIONS)
    plt.title(f"{model_name} - Normalized Confusion Matrix")
    plt.xlabel("Predicted Emotion")
    plt.ylabel("True Emotion")
    plt.tight_layout()
    plt.savefig(f"{model_name.lower()}_confusion_matrix.png")
    plt.show()
    return acc, macro_f1

mb_acc, mb_f1 = evaluate_and_plot(mobilenet, mb_test_loader, "MobileNetV3")
vit_acc, vit_f1 = evaluate_and_plot(vit_model, vit_test_loader, "VisionTransformer")

# Export the winner to checkpoints/face_model
os.makedirs("checkpoints/face_model", exist_ok=True)
if vit_f1 >= mb_f1:
    print("Vision Transformer achieved superior Macro-F1. Exporting ViT checkpoint...")
    vit_model.save_pretrained("checkpoints/face_model")
else:
    print("MobileNetV3 achieved superior Macro-F1. Exporting MobileNet checkpoint...")
    torch.save(mobilenet.state_dict(), "checkpoints/face_model/model.pt")